In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, validation_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle

print(f"Pandas: {version('pandas')}")
print(f"Numpy: {version('numpy')}")
print(f"Matplotlib: {version('matplotlib')}")
print(f"Seaborn: {version('seaborn')}")
print(f"Scikit-learn: {version('scikit-learn')}")


Pandas: 3.0.0
Numpy: 1.26.4
Matplotlib: 3.10.8
Seaborn: 0.13.2
Scikit-learn: 1.8.0


In [2]:
data = pd.read_csv('../L1 Regresion Lineal/data/Datos Lab 1.csv')
data = data.copy()

print("=" * 70)
print("Preparación de datos")
print("=" * 70)
print(f"Registros iniciales: {data.shape[0]}")

duplicados = data.duplicated().sum()
print(f"Duplicados: {duplicados}")
data = data.drop_duplicates()
print(f"Registros después de eliminar duplicados: {data.shape[0]}")

data = data.dropna(subset=['CVD Risk Score'])
print(f"Registros después de eliminar filas con NA en 'CVD Risk Score': {data.shape[0]}")

print("Sección de variables:")
variables_objetivo = 'CVD Risk Score'
variables_excluir = ['Patient ID', 'Date of Service', 'Blood Pressure (mmHg)', 'Height (m)', 'Height (cm)', 'CVD Risk Level']
print(f"Variables excluidas: {variables_excluir}")

X = data.drop(columns=variables_excluir + [variables_objetivo])
y = data[variables_objetivo]

print(f"Variables seleccionadas para el modelo: {X.shape[1]}")
print(X.columns.tolist())

variables_numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
variables_categoricas = X.select_dtypes(include=['object']).columns.tolist()
print(f"Variables numericas ({len(variables_numericas)}): {variables_numericas}")
print(f"Variables categoricas ({len(variables_categoricas)}): {variables_categoricas}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Tamaño conjunto entrenamiento: {X_train.shape[0]} ({100*X_train.shape[0]/X.shape[0]:.1f}%)")
print(f"Tamaño conjunto prueba: {X_test.shape[0]} ({100*X_test.shape[0]/X.shape[0]:.1f}%)")

preprocesamiento_completo = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputador', SimpleImputer(strategy='median')),
            ('escalador', StandardScaler())
            
        ]),variables_numericas),
        ('cat', Pipeline(steps=[
            ('imputador', SimpleImputer(strategy='most_frequent')),
            ('codificador', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]),variables_categoricas)
    ])

X_train_prep = preprocesamiento_completo.fit_transform(X_train)
X_test_prep = preprocesamiento_completo.transform(X_test)

print(f"Dimensiones despues del procesamiento: {X_train_prep.shape}")




Preparación de datos
Registros iniciales: 1639
Duplicados: 151
Registros después de eliminar duplicados: 1488
Registros después de eliminar filas con NA en 'CVD Risk Score': 1460
Sección de variables:
Variables excluidas: ['Patient ID', 'Date of Service', 'Blood Pressure (mmHg)', 'Height (m)', 'Height (cm)', 'CVD Risk Level']
Variables seleccionadas para el modelo: 17
['Sex', 'Age', 'Weight (kg)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Blood Pressure Category', 'Estimated LDL (mg/dL)']
Variables numericas (11): ['Age', 'Weight (kg)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Variables categoricas (6): ['Sex', 'Smoking 

In [3]:
print("=" * 70)
print("Modelo 1: Regresión Polinomial")
print("=" * 70)

prep_poly = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler())
])

X_train_poly = prep_poly.fit_transform(X_train[variables_numericas])
X_test_poly = prep_poly.transform(X_test[variables_numericas])

param_grid_poly = {
    'poly_degree': [1, 2, 3, 4, 5],
}

resultados_poly = []

for degree in param_grid_poly['poly_degree']:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly_feat = poly.fit_transform(X_train_poly)
    X_test_poly_feat = poly.transform(X_test_poly)
    
    modelo = LinearRegression()
    modelo.fit(X_train_poly_feat, y_train)
    
    y_pred_train = modelo.predict(X_train_poly_feat)
    y_pred_test = modelo.predict(X_test_poly_feat)
    
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    
    cv_resultados = cross_val_score(LinearRegression(), X_train_poly_feat, y_train, cv=5, scoring='neg_mean_squared_error')
    rmse_cv = np.sqrt(-cv_resultados.mean())
    rmse_cv_std = np.sqrt(-cv_resultados.std())
    
    resultados_poly.append({
        'degree': degree,
        'n_features': X_train_poly_feat.shape[1],
        'rmse_train': rmse_train,
        'rmse_test': rmse_test,
        'mae_train': mae_train,
        'mae_test': mae_test,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'rmse_cv': rmse_cv,
        'rmse_cv_std': rmse_cv_std,
        'modelo': modelo,
        'poly': poly,
        'X_train_poly_feat': X_train_poly_feat,
        'X_test_poly_feat': X_test_poly_feat
    })
    
df_poly = pd.DataFrame([{
    'Grado': r['degree'],
    'Características': r['n_features'],
    'RMSE Entrenamiento': r['rmse_train'],
    'RMSE Prueba': r['rmse_test'],
    'MAE Entrenamiento': r['mae_train'],
    'MAE Prueba': r['mae_test'],
    'R² Entrenamiento': r['r2_train'],
    'R² Prueba': r['r2_test'],
    'RMSE CV media': r['rmse_cv'],
    'RMSE CV Std': r['rmse_cv_std']
} for r in resultados_poly])

print("\nResultados de Regresión Polinomial:")
print(df_poly.to_string(index=False))

mejor_poly = resultados_poly[np.argmin([r['rmse_cv'] for r in resultados_poly])]
print(f"\nMejor grado polinomial: {mejor_poly['degree']} (RMSE CV: {mejor_poly['rmse_cv']:.4f})")


Modelo 1: Regresión Polinomial

Resultados de Regresión Polinomial:
 Grado  Características  RMSE Entrenamiento  RMSE Prueba  MAE Entrenamiento  MAE Prueba  R² Entrenamiento   R² Prueba  RMSE CV media  RMSE CV Std
     1               11           10.695619    10.765521           3.513084    3.683505          0.033604    0.004376      10.863488          NaN
     2               77           10.336098    11.430409           4.068764    4.599749          0.097481   -0.122404      11.197702          NaN
     3              363            8.942892    17.488738           4.545185    8.988398          0.324385   -1.627498      40.501418          NaN
     4             1364            2.786322   327.751385           0.649573  130.629033          0.934415 -921.814817      93.884639          NaN
     5             4367            2.786322    45.167475           0.649573   23.394382          0.934415  -16.525770      48.187142          NaN

Mejor grado polinomial: 1 (RMSE CV: 10.8635)
